In [2]:
!pip install datasets

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.4/485.4 KB 4.3 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 KB 318.5 kB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 KB 7.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 KB 8.9 MB/s eta 0:00:00


In [7]:
import os
import torch 
import logging
import pandas as pd
import numpy as np
from PIL import Image
import torch.nn as nn 
from datasets import load_dataset, load_from_disk
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms 

In [5]:
dataset_with_butterflies = load_from_disk('/home/nikolay/thesis/tiger-fox-elephant/ddpm/src/thesis/tiger-fox-elephant/ddpm/src/data/butterflies_dataset')
print(dataset_with_butterflies)

DatasetDict({
    train: Dataset({
        features: ['image_url', 'image_alt', 'id', 'name', 'scientific_name', 'gender', 'taxonomy', 'region', 'locality', 'date', 'usnm_no', 'guid', 'edan_url', 'source', 'stage', 'image', 'image_hash', 'sim_score'],
        num_rows: 1000
    })
})


In [9]:
df = pd.DataFrame(dataset_with_butterflies['train'])
df.head()

,image_url,image_alt,id,name,scientific_name,gender,taxonomy,region,locality,date,usnm_no,guid,edan_url,source,stage,image,image_hash,sim_score
0,https://ids.si.edu/ids/deliveryService?id=ark:...,view Paper Kite digital asset number 1,ark:/65665/m3b302800a43ef46b8a7a87a7b1eb06ab6,Paper Kite,Idea leuconoe,None,"Animalia, Arthropoda, Hexapoda, Insecta, Lepid...","US Mid Atlantic (PA, NJ, MD, DE, DC, VA, WV)","NMNH Butterfly Pavilion, North America, United...",None,EO401041,http://n2t.net/ark:/65665/35f90bc1d-2e3c-4798-...,edanmdm:nmnheducation_11038234,Smithsonian Education and Outreach collections,None,<PIL.PngImagePlugin.PngImageFile image mode=RG...,fb0b8749d437efc70a26e54212b3572c,0.805520
1,https://ids.si.edu/ids/deliveryService?id=ark:...,view Doris Longwing digital asset number 1,ark:/65665/m351f75857f81840acb01d0eb82f2a1784,Doris Longwing,Heliconius doris,None,"Animalia, Arthropoda, Hexapoda, Insecta, Lepid...","US Mid Atlantic (PA, NJ, MD, DE, DC, VA, WV)","NMNH Insect Zoo, North America, United States,...",None,EO401027,http://n2t.net/ark:/65665/3f4d0cc10-f3b4-4323-...,edanmdm:nmnheducation_11038220,Smithsonian Education and Outreach collections,None,<PIL.PngImagePlugin.PngImageFile image mode=RG...,9657726e69494021d1c9929ee7b375fa,0.810842
2,https://ids.si.edu/ids/deliveryService?id=ark:...,"view Asian Swallowtail, Chinese Yellow Swallow...",ark:/65665/m3ac764368a66846179e863101219b6fec,"Asian Swallowtail, Chinese Yellow Swallowtail",Papilio xuthus,None,"Animalia, Arthropoda, Hexapoda, Insecta, Lepid...","US Mid Atlantic (PA, NJ, MD, DE, DC, VA, WV)","NMNH Butterfly Pavilion, North America, United...",None,EO401045,http://n2t.net/ark:/65665/33446acdf-f0fd-44f0-...,edanmdm:nmnheducation_11038238,Smithsonian Education and Outreach collections,None,<PIL.PngImagePlugin.PngImageFile image mode=RG...,9303ab0ac75fd0d3047ba987d268f871,0.813563
3,https://ids.si.edu/ids/deliveryService?id=ark:...,view Postman digital asset number 1,ark:/65665/m3855802e6e88c48c58eec449d6e811677,Postman,Heliconius melpomene sticheli,None,"Animalia, Arthropoda, Hexapoda, Insecta, Lepid...","US Mid Atlantic (PA, NJ, MD, DE, DC, VA, WV)","NMNH Insect Zoo, North America, United States,...",None,EO401024,http://n2t.net/ark:/65665/359f1f2d7-bd05-4964-...,edanmdm:nmnheducation_11038217,Smithsonian Education and Outreach collections,None,<PIL.PngImagePlugin.PngImageFile image mode=RG...,3726a5faf63c3d70db5d433705b53ba9,0.813736
4,https://ids.si.edu/ids/deliveryService?id=ark:...,"view Red Postman, Small Postman digital asset ...",ark:/65665/m3e3f4435a245b4a9d9e94a80bd5c0d4a9,"Red Postman, Small Postman",Heliconius erato notabilis,None,"Animalia, Arthropoda, Hexapoda, Insecta, Lepid...","US Mid Atlantic (PA, NJ, MD, DE, DC, VA, WV)","NMNH Insect Zoo, North America, United States,...",None,EO401021,http://n2t.net/ark:/65665/3f50ca15b-8f27-45c7-...,edanmdm:nmnheducation_11038213,Smithsonian Education and Outreach collections,None,<PIL.PngImagePlugin.PngImageFile image mode=RG...,3aa4d93629910b3fe1165c4fc20033fc,0.814460


In [11]:
df = df[['image_url', 'id', 'name']]

In [12]:
df.head()

,image_url,id,name
0,https://ids.si.edu/ids/deliveryService?id=ark:...,ark:/65665/m3b302800a43ef46b8a7a87a7b1eb06ab6,Paper Kite
1,https://ids.si.edu/ids/deliveryService?id=ark:...,ark:/65665/m351f75857f81840acb01d0eb82f2a1784,Doris Longwing
2,https://ids.si.edu/ids/deliveryService?id=ark:...,ark:/65665/m3ac764368a66846179e863101219b6fec,"Asian Swallowtail, Chinese Yellow Swallowtail"
3,https://ids.si.edu/ids/deliveryService?id=ark:...,ark:/65665/m3855802e6e88c48c58eec449d6e811677,Postman
4,https://ids.si.edu/ids/deliveryService?id=ark:...,ark:/65665/m3e3f4435a245b4a9d9e94a80bd5c0d4a9,"Red Postman, Small Postman"


In [14]:
import os
import requests
from PIL import Image
from io import BytesIO

def save_images_from_urls(df, output_path):
    output_dir = os.path.join(output_path, 'images')
    
    os.makedirs(output_dir, exist_ok=True)

    for i, row in df.iterrows():
        try:
            response = requests.get(row['image_url'], stream=True, timeout=10)
            response.raise_for_status() 
            
            image = Image.open(BytesIO(response.content))
            
            file_id = row['id'].split('/')[-1] 
            filename = os.path.join(output_dir, f'{file_id}.jpg')
            
            if image.mode != 'RGB':
                image = image.convert('RGB')
                
            image.save(filename)
            print(f'Saved: {filename}')
            
        except Exception as e:
            print(f'Error with {row["image_url"]}: {str(e)}')

save_images_from_urls(df, output_path='/home/nikolay/thesis/tiger-fox-elephant/ddpm/src/data')

Saved: /home/nikolay/thesis/tiger-fox-elephant/ddpm/src/data/images/m3b302800a43ef46b8a7a87a7b1eb06ab6.jpg
Saved: /home/nikolay/thesis/tiger-fox-elephant/ddpm/src/data/images/m351f75857f81840acb01d0eb82f2a1784.jpg
Saved: /home/nikolay/thesis/tiger-fox-elephant/ddpm/src/data/images/m3ac764368a66846179e863101219b6fec.jpg
Saved: /home/nikolay/thesis/tiger-fox-elephant/ddpm/src/data/images/m3855802e6e88c48c58eec449d6e811677.jpg
Saved: /home/nikolay/thesis/tiger-fox-elephant/ddpm/src/data/images/m3e3f4435a245b4a9d9e94a80bd5c0d4a9.jpg
Saved: /home/nikolay/thesis/tiger-fox-elephant/ddpm/src/data/images/m32663c77db7964deb9afb658b6ae0c9f1.jpg
Saved: /home/nikolay/thesis/tiger-fox-elephant/ddpm/src/data/images/m37bed76876a8a4f388ff496cf165368f6.jpg
Saved: /home/nikolay/thesis/tiger-fox-elephant/ddpm/src/data/images/m369e4727a4fe8423ebc4c93b8241cd493.jpg
Saved: /home/nikolay/thesis/tiger-fox-elephant/ddpm/src/data/images/m3b2a7387f6ad8446cb680c0dd973d3ff6.jpg
Saved: /home/nikolay/thesis/tiger-fox